# Random Forest


---
## 1. 🎯 Motivação — O Problema das Árvores Individuais

Uma **Árvore de Decisão** cresce aprendendo regras de particionamento dos dados de treino. Quando deixamos a árvore crescer livremente:

- Ela memoriza o ruído do treino → **overfitting** severo
- Alta **variância**: pequenas mudanças nos dados geram árvores completamente diferentes
- Baixo **viés**, mas generalização ruim

### O dilema Viés-Variância

| Modelo         | Viés  | Variância | Problema         |
|----------------|-------|-----------|------------------|
| Árvore rasa    | Alto  | Baixa     | Underfitting     |
| Árvore profunda| Baixo | Alta      | Overfitting      |
| **Random Forest**  | **Baixo** | **Baixa** | ✅ Equilíbrio |

> **Insight-chave:** Se combinarmos muitas árvores com *alta variância mas baixo viés*, e garantirmos que elas sejam **diversas entre si**, a média delas cancela o ruído individual.

---
## 2. 🧩 Ensemble Learning

**Ensemble** = combinar múltiplos modelos para obter desempenho superior ao de qualquer modelo individual.

### Analogia: O Júri
Imagine 100 jurados votando independentemente sobre um caso. Mesmo que cada um erre 30% das vezes, a decisão por maioria estará correta com probabilidade muito superior a 70% — desde que os erros sejam **descorrelacionados**.

### Tipos de Ensemble

| Técnica    | Ideia                                      | Exemplo         |
|------------|--------------------------------------------|-----------------|
| **Bagging**    | Treina modelos em subconjuntos aleatórios  | Random Forest   |
| **Boosting**   | Treina sequencialmente, corrigindo erros   | XGBoost, AdaBoost |
| **Stacking**   | Usa um meta-modelo para combinar predições | Stacking genérico |

Random Forest usa **Bagging** (+ aleatoriedade extra nas features).

---
## 3. 🎲 Bagging e Bootstrap Sampling

**Bootstrap Sampling** = amostrar *com reposição* um conjunto de tamanho n a partir dos n exemplos originais.

- Cada amostra bootstrap tem ~63,2% dos dados originais (únicos)
- Os ~36,8% restantes formam o **Out-of-Bag (OOB)** — conjunto de validação gratuito!

**Bagging** (Bootstrap Aggregating):
1. Gere B amostras bootstrap
2. Treine um modelo base em cada amostra
3. Agregue as predições:
   - Classificação → votação majoritária
   - Regressão → média aritmética

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─── Demonstração: Bootstrap Sampling ───────────────────────────────────────
np.random.seed(42)
n = 10
dados_originais = np.arange(1, n + 1)

print("Dados originais:", dados_originais)
print()

for i in range(3):
    amostra = np.random.choice(dados_originais, size=n, replace=True)
    oob     = set(dados_originais) - set(amostra)
    print(f"Bootstrap {i+1}: {sorted(amostra)}  |  OOB: {sorted(oob)}")

# Proporção teórica de amostras únicas
prob_unica = (1 - (1 - 1/n)**n)
print(f"\nProporção esperada de únicos por bootstrap: {prob_unica:.3f}")
print(f"(1 - 1/e ≈ {1 - 1/np.e:.3f})")

---
## 4. 🌲🌲🌲 Random Forest: Algoritmo Completo

Random Forest acrescenta ao Bagging uma segunda fonte de aleatoriedade: **seleção aleatória de features**.

### Algoritmo de Treinamento

```
Para b = 1 até B:
  1. Gere amostra bootstrap Z_b de tamanho n
  2. Construa árvore T_b a partir de Z_b:
     Para cada nó:
       a. Selecione m features aleatoriamente (m << p)
       b. Escolha a melhor divisão dentre essas m features
       c. Divida o nó em dois filhos
       d. Repita até critério de parada

Retorne ensemble {T_1, ..., T_B}
```

### Predição

**Classificação:**  
$\hat{y} = \text{moda}\{T_1(x), T_2(x), ..., T_B(x)\}$

**Regressão:**  
$\hat{y} = \dfrac{1}{B} \sum_{b=1}^{B} T_b(x)$

### Por que selecionar features aleatoriamente?

Sem isso, se uma feature é muito preditiva, todas as árvores a usarão primeiro → árvores **correlacionadas** → a média não reduz a variância efetivamente.

Com `m` features aleatórias por nó:
- Árvores se tornam **descorrelacionadas**
- A redução de variância pelo ensemble é maximizada

### Valores típicos de m

| Tarefa         | Regra padrão              |
|----------------|---------------------------|
| Classificação  | $m = \sqrt{p}$            |
| Regressão      | $m = p/3$                 |

---
## 5. 📊 Importância de Features

Random Forest fornece, naturalmente, uma medida de importância para cada feature:

### Mean Decrease in Impurity (MDI)
Para cada árvore, para cada nó onde a feature $j$ é usada:
$$\text{Importância}(j) = \sum_{b=1}^{B} \sum_{\text{nós com } j} \Delta \text{Impureza}(\text{nó}) \times \frac{n_{\text{nó}}}{n}$$

Normalizado para que a soma seja 1.

### Mean Decrease in Accuracy (MDA / Permutation Importance)
1. Calcule o erro OOB base
2. Para cada feature $j$: embaralhe seus valores nas amostras OOB
3. Mede o aumento no erro OOB → quanto a feature contribui

---
## 6. ⚙️ Hiperparâmetros Principais (scikit-learn)

| Parâmetro             | Descrição                                         | Valor padrão |
|-----------------------|---------------------------------------------------|--------------|
| `n_estimators`        | Número de árvores B                               | 100          |
| `max_depth`           | Profundidade máxima de cada árvore                | None (livre) |
| `max_features`        | Número m de features por nó                      | `'sqrt'`     |
| `min_samples_split`   | Mínimo de amostras para dividir um nó             | 2            |
| `min_samples_leaf`    | Mínimo de amostras em folha                       | 1            |
| `oob_score`           | Usa OOB como estimativa do erro                   | False        |
| `n_jobs`              | Paralelismo (-1 = todos os cores)                 | None         |
| `random_state`        | Semente para reprodutibilidade                    | None         |

**Dica prática:**
- Mais árvores = melhor (até convergir); `n_estimators ≥ 100` é seguro
- `max_depth` e `min_samples_leaf` controlam overfitting
- Use `oob_score=True` como substituto rápido de cross-validation

---
## 7. 💻 Exemplo Prático — Classificação

### Dataset: Iris (3 classes de flores)

In [ ]:
# Instalação (apenas se necessário no Colab)
# !pip install scikit-learn matplotlib seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_diabetes
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, mean_squared_error, r2_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Bibliotecas carregadas com sucesso!")

In [ ]:
# ─── Carregando e explorando o dataset ──────────────────────────────────────
iris = load_iris(as_frame=True)
df   = iris.frame
df['target_name'] = iris.target_names[iris.target]

print("Shape:", df.shape)
print("\nDistribuição das classes:")
print(df['target_name'].value_counts())
df.head()

In [ ]:
# ─── Visualização exploratória ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
colors = {'setosa': '#2196F3', 'versicolor': '#FF5722', 'virginica': '#4CAF50'}
for especie, cor in colors.items():
    mask = df['target_name'] == especie
    axes[0].scatter(df.loc[mask, 'petal length (cm)'],
                    df.loc[mask, 'petal width (cm)'],
                    label=especie, color=cor, alpha=0.7, s=60)
axes[0].set_xlabel('Comprimento da Pétala (cm)')
axes[0].set_ylabel('Largura da Pétala (cm)')
axes[0].set_title('Iris: Pétala')
axes[0].legend()

# Pairplot simplificado
for especie, cor in colors.items():
    mask = df['target_name'] == especie
    axes[1].scatter(df.loc[mask, 'sepal length (cm)'],
                    df.loc[mask, 'sepal width (cm)'],
                    label=especie, color=cor, alpha=0.7, s=60)
axes[1].set_xlabel('Comprimento da Sépala (cm)')
axes[1].set_ylabel('Largura da Sépala (cm)')
axes[1].set_title('Iris: Sépala')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ─── Treinamento ─────────────────────────────────────────────────────────────
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Árvore individual (baseline)
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

print("=" * 45)
print(f"  Árvore Individual — Acurácia Teste: {dt.score(X_test, y_test):.4f}")
print(f"  Random Forest     — Acurácia Teste: {rf.score(X_test, y_test):.4f}")
print(f"  Random Forest     — OOB Score:      {rf.oob_score_:.4f}")
print("=" * 45)

In [ ]:
# ─── Métricas completas ───────────────────────────────────────────────────────
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Matriz de confusão
fig, ax = plt.subplots(figsize=(6, 5))
cm  = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de Confusão — Random Forest (Iris)')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Importância de Features ─────────────────────────────────────────────────
feature_names  = iris.feature_names
importancias   = rf.feature_importances_
indices        = np.argsort(importancias)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MDI
axes[0].barh([feature_names[i] for i in indices],
             importancias[indices],
             color='#1976D2')
axes[0].set_xlabel('Importância (MDI)')
axes[0].set_title('Importância das Features\n(Mean Decrease in Impurity)')
axes[0].invert_yaxis()

# Permutation Importance
perm = permutation_importance(rf, X_test, y_test, n_repeats=30, random_state=42)
sorted_idx = perm.importances_mean.argsort()[::-1]
bp = axes[1].boxplot(
    perm.importances[sorted_idx].T,
    vert=False,
    labels=[feature_names[i] for i in sorted_idx]
)
axes[1].set_xlabel('Importância (Permutation)')
axes[1].set_title('Importância das Features\n(Permutation Importance)')

plt.tight_layout()
plt.show()

print("\nAs pétalas (comprimento e largura) dominam a classificação da íris! ✅")

---
## 7b. 💻 Exemplo Prático — Regressão

### Dataset: Diabetes (predição de progressão da doença)

In [ ]:
# ─── Dataset Diabetes ────────────────────────────────────────────────────────
diabetes = load_diabetes(as_frame=True)
Xr, yr   = diabetes.data, diabetes.target

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)

rfr = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rfr.fit(Xr_train, yr_train)

yr_pred = rfr.predict(Xr_test)
rmse    = np.sqrt(mean_squared_error(yr_test, yr_pred))
r2      = r2_score(yr_test, yr_pred)

print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")
print(f"OOB R²: {rfr.oob_score_:.4f}")

# Plot predito vs real
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(yr_test, yr_pred, alpha=0.5, color='#E91E63', edgecolors='white', s=50)
lim = [yr_test.min(), yr_test.max()]
ax.plot(lim, lim, 'k--', lw=1.5, label='Ideal')
ax.set_xlabel('Valor Real')
ax.set_ylabel('Valor Predito')
ax.set_title(f'Predito vs Real — Random Forest Regressão\nR² = {r2:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. 🔍 Efeito do Número de Árvores e Profundidade

In [ ]:
# ─── Efeito de n_estimators ──────────────────────────────────────────────────
n_vals    = [1, 5, 10, 20, 50, 100, 200, 300]
acc_train = []
acc_test  = []

for n in n_vals:
    modelo = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    modelo.fit(X_train, y_train)
    acc_train.append(modelo.score(X_train, y_train))
    acc_test.append(modelo.score(X_test, y_test))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(n_vals, acc_train, 'o-', label='Treino', color='#FF5722')
axes[0].plot(n_vals, acc_test,  's-', label='Teste',  color='#1976D2')
axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('Acurácia')
axes[0].set_title('Efeito do Número de Árvores')
axes[0].legend()
axes[0].set_ylim([0.8, 1.02])

# Efeito de max_depth
depths     = [1, 2, 3, 4, 5, None]
acc_train2 = []
acc_test2  = []
labels_d   = [str(d) if d else 'None' for d in depths]

for d in depths:
    modelo = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42)
    modelo.fit(X_train, y_train)
    acc_train2.append(modelo.score(X_train, y_train))
    acc_test2.append(modelo.score(X_test, y_test))

x_ = range(len(depths))
axes[1].plot(x_, acc_train2, 'o-', label='Treino', color='#FF5722')
axes[1].plot(x_, acc_test2,  's-', label='Teste',  color='#1976D2')
axes[1].set_xticks(x_)
axes[1].set_xticklabels(labels_d)
axes[1].set_xlabel('max_depth')
axes[1].set_ylabel('Acurácia')
axes[1].set_title('Efeito da Profundidade Máxima')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 9. 🔎 Fronteira de Decisão (Visualização 2D)

Para visualizar a fronteira, usamos apenas 2 features (pétala).

In [ ]:
from matplotlib.colors import ListedColormap

# Usa apenas as 2 features de pétala
X2  = iris.data[:, 2:4]
y2  = iris.target

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.3, random_state=42, stratify=y2
)

modelos = [
    ("Árvore de Decisão",   DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("Random Forest (5)",   RandomForestClassifier(n_estimators=5,   random_state=42)),
    ("Random Forest (100)", RandomForestClassifier(n_estimators=100, random_state=42)),
]

h   = 0.02
x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

cmap_bg = ListedColormap(['#BBDEFB', '#FFCCBC', '#C8E6C9'])
cmap_pt = ListedColormap(['#1565C0', '#BF360C', '#1B5E20'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nome, modelo) in zip(axes, modelos):
    modelo.fit(X2_train, y2_train)
    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
    sc = ax.scatter(X2_test[:, 0], X2_test[:, 1], c=y2_test,
                    cmap=cmap_pt, edgecolors='white', s=50, linewidths=0.5)
    acc = modelo.score(X2_test, y2_test)
    ax.set_title(f"{nome}\nAcurácia: {acc:.3f}")
    ax.set_xlabel('Comprimento da Pétala')
    ax.set_ylabel('Largura da Pétala')

plt.suptitle('Fronteiras de Decisão: Árvore vs. Random Forest', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Cross-Validation e Grid Search

In [ ]:
# ─── Cross-Validation 5-fold ─────────────────────────────────────────────────
rf_cv = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
scores = cross_val_score(rf_cv, X, y, cv=5, scoring='accuracy')

print("Acurácias por fold:", np.round(scores, 4))
print(f"Média:  {scores.mean():.4f} ± {scores.std():.4f}")

# ─── Grid Search ────────────────────────────────────────────────────────────
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [3, 5, None],
    'max_features': ['sqrt', 'log2'],
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
gs.fit(X_train, y_train)

print("\n🏆 Melhores hiperparâmetros:", gs.best_params_)
print(f"   Melhor CV Score: {gs.best_score_:.4f}")
print(f"   Acurácia Teste:  {gs.best_estimator_.score(X_test, y_test):.4f}")

---
## ✏️ Exercícios

### Exercício 1 — Análise Comparativa de Modelos

Carregue o dataset `load_wine()` do scikit-learn e:

a) Divida em treino (75%) e teste (25%) com `stratify=y` e `random_state=0`.  
b) Treine um `DecisionTreeClassifier`, um `RandomForestClassifier(n_estimators=100)` e um `RandomForestClassifier(n_estimators=500)`.  
c) Reporte a acurácia de cada modelo no conjunto de teste.  
d) Para o Random Forest de 500 árvores, imprima o `oob_score_`.  
e) **Questão de análise:** O OOB Score é uma estimativa otimista ou pessimista da acurácia no teste? Justifique.

In [ ]:
# ── Exercício 1 ── Escreva seu código aqui ───────────────────────────────────
from sklearn.datasets import load_wine

# Seu código:


---
### Exercício 2 — Importância de Features no Mundo Real

Use o dataset `load_breast_cancer()` (diagnóstico de câncer de mama):

a) Treine um `RandomForestClassifier` com `n_estimators=200` e `random_state=42`.  
b) Plote as 10 features mais importantes (MDI) em um gráfico de barras horizontal.  
c) Calcule também a **Permutation Importance** para as mesmas features.  
d) Compare as duas listas: há diferenças na ordem? Quando pode haver divergência entre MDI e Permutation Importance?

In [ ]:
# ── Exercício 2 ── Escreva seu código aqui ───────────────────────────────────
from sklearn.datasets import load_breast_cancer

# Seu código:


---
### Exercício 3 — Curva de Convergência

Usando o dataset Iris:

a) Treine Random Forests com `n_estimators` variando de 1 a 200 (passo de 5).  
b) Para cada valor, calcule o **erro OOB** (use `oob_score=True` e converta para erro: `1 - oob_score_`).  
c) Plote a curva Erro OOB × n_estimators.  
d) A partir de qual número de árvores o erro parece estabilizar? Existe custo em usar mais árvores que o necessário?

In [ ]:
# ── Exercício 3 ── Escreva seu código aqui ───────────────────────────────────

# Seu código:


---
### Exercício 4 — Desafio: Regressão com Dados Sintéticos

a) Gere dados sintéticos com `make_regression(n_samples=500, n_features=10, noise=30, random_state=42)`.  
b) Treine um `RandomForestRegressor` e ajuste `max_depth` entre 2 e 20 usando GridSearchCV com CV=5 e scoring `'neg_mean_squared_error'`.  
c) Plote o RMSE de validação × max_depth.  
d) Qual `max_depth` minimizou o RMSE?  
e) **Bônus:** Adicione `min_samples_leaf` como segundo hiperparâmetro no Grid Search.

In [ ]:
# ── Exercício 4 ── Escreva seu código aqui ───────────────────────────────────
from sklearn.datasets import make_regression

# Seu código:


---
## 💡 Gabarito Comentado

> **Atenção:** Tente resolver os exercícios antes de consultar! 👇

In [ ]:
# ── Gabarito Exercício 1 ─────────────────────────────────────────────────────
from sklearn.datasets import load_wine

wine = load_wine()
Xw, yw = wine.data, wine.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.25,
                                                stratify=yw, random_state=0)

modelos_wine = [
    ('Árvore',    DecisionTreeClassifier(random_state=42)),
    ('RF-100',    RandomForestClassifier(n_estimators=100, oob_score=True,  random_state=42)),
    ('RF-500',    RandomForestClassifier(n_estimators=500, oob_score=True,  random_state=42)),
]

for nome, mod in modelos_wine:
    mod.fit(Xw_tr, yw_tr)
    oob = f"  OOB: {mod.oob_score_:.4f}" if hasattr(mod, 'oob_score_') else ""
    print(f"{nome:12s}  Teste: {mod.score(Xw_te, yw_te):.4f}{oob}")

print("""
Análise: O OOB Score usa ~36% dos dados não vistos em cada árvore — é uma estimativa
não-enviesada do erro de generalização. Pode diferir ligeiramente do score no conjunto
de teste fixo, mas tende a ser próximo. Não é sistematicamente otimista nem pessimista.
""")

In [ ]:
# ── Gabarito Exercício 2 ─────────────────────────────────────────────────────
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
Xb, yb = bc.data, bc.target
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.25, random_state=42)

rf_bc = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_bc.fit(Xb_tr, yb_tr)

# MDI
imp_mdi = pd.Series(rf_bc.feature_importances_, index=bc.feature_names).nlargest(10)

# Permutation
perm_bc  = permutation_importance(rf_bc, Xb_te, yb_te, n_repeats=20, random_state=42)
imp_perm = pd.Series(perm_bc.importances_mean, index=bc.feature_names).nlargest(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imp_mdi.sort_values().plot(kind='barh', ax=axes[0], color='#1976D2')
axes[0].set_title('Top 10 Features — MDI')
imp_perm.sort_values().plot(kind='barh', ax=axes[1], color='#E91E63')
axes[1].set_title('Top 10 Features — Permutation')
plt.tight_layout(); plt.show()

print("MDI top-3:        ", list(imp_mdi.index[:3]))
print("Permutation top-3:", list(imp_perm.index[:3]))

In [ ]:
# ── Gabarito Exercício 3 ─────────────────────────────────────────────────────
n_range = range(1, 201, 5)
erros_oob = []

for n in n_range:
    m = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42)
    m.fit(X_train, y_train)
    erros_oob.append(1 - m.oob_score_)

plt.figure(figsize=(8, 4))
plt.plot(list(n_range), erros_oob, '-o', markersize=3, color='#1976D2')
plt.xlabel('n_estimators')
plt.ylabel('Erro OOB')
plt.title('Convergência do Erro OOB com o Número de Árvores')
plt.tight_layout()
plt.show()

print("Comentário: O erro OOB cai rapidamente e converge por volta de 50-100 árvores.")
print("Usar mais árvores não piora o modelo, mas aumenta o custo computacional.")

In [ ]:
# ── Gabarito Exercício 4 ─────────────────────────────────────────────────────
from sklearn.datasets import make_regression

Xsyn, ysyn = make_regression(n_samples=500, n_features=10, noise=30, random_state=42)
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xsyn, ysyn, test_size=0.2, random_state=42)

param_r = {
    'max_depth': list(range(2, 21)),
    'min_samples_leaf': [1, 3, 5, 10],   # bônus
}

gs_r = GridSearchCV(
    RandomForestRegressor(n_estimators=100, random_state=42),
    param_r, cv=5, scoring='neg_mean_squared_error', n_jobs=-1
)
gs_r.fit(Xs_tr, ys_tr)

print("Melhores parâmetros:", gs_r.best_params_)

# Curva max_depth (fixando min_samples_leaf=1 para visualização)
cv_res = pd.DataFrame(gs_r.cv_results_)
mask   = cv_res['param_min_samples_leaf'] == 1
sub    = cv_res[mask].copy()
sub['rmse_cv'] = np.sqrt(-sub['mean_test_score'])
sub = sub.sort_values('param_max_depth')

plt.figure(figsize=(8, 4))
plt.plot(sub['param_max_depth'], sub['rmse_cv'], 'o-', color='#7B1FA2')
plt.xlabel('max_depth')
plt.ylabel('RMSE (validação cruzada)')
plt.title('RMSE × max_depth (min_samples_leaf=1)')
plt.tight_layout()
plt.show()

---
## 📌 Resumo da Aula

| Conceito            | O que aprendemos                                                |
|---------------------|-----------------------------------------------------------------|
| **Bagging**         | Treinar modelos em amostras bootstrap → reduz variância         |
| **Feature Aleatória**| Selecionar m features por nó → descorrelaciona as árvores      |
| **OOB Score**       | Estimativa gratuita do erro de generalização                    |
| **Importância MDI** | Redução média de impureza ao longo de todas as árvores         |
| **Permutation**     | Importância avaliada na prática (mais robusta para colineridade)|
| **n_estimators**    | Mais árvores = mais estável, até convergência                   |
| **max_depth**       | Controla o tradeoff viés-variância de cada árvore              |

### Próxima Aula
**🚀 Random Forest com Gradient Boosting** — como treinar sequencialmente para corrigir erros residuais e obter ainda mais poder preditivo.
